**Avvio locale e Colab:** preparare la cartella completa seguendo il [README del laboratorio](README.md). Conservare gli identificativi delle celle e salvare gli output nel proprio runtime.

<!-- C01 -->
# Computer Vision · Lezione 13
## Dalle CNN ai transformer
26 ottobre 2026. **Notebook dimostrativo completo**, senza addestramento. Tutte le **17 celle di codice** sono già scritte. C01–C27 studiano operatori con piccoli tensori assegnati in float64; C28–C35 calcolano una mappa CLS-verso-patch reale con DeiT Tiny preaddestrato, su CPU in float32.

Per la demo fotografica aprire il notebook insieme alle cartelle `supporto`, `input` e `pesi` del pacchetto. I pesi e la foto sono già inclusi: non occorrono download durante lo studio. Per Colab seguire `Avvio_Colab.md` e caricare lo ZIP completo nello stesso runtime.

N01–N15 sono letture guidate con spiegazioni nello stesso notebook; F01–F05 sono una sintesi già svolta.  e  secondo le istruzioni del relativo notebook.

Nei materiali del corso si usa il **punto decimale** (0.5) e la **virgola seguita da uno spazio** fra componenti di vettori, per esempio (1.2, 1, −0.3).

<!-- C02 -->
## Percorso e convenzioni
Le etichette Cxx contano tutte le celle, comprese queste due. Se il kernel è nuovo, eseguire prima C03. C13 precede C15 e C15 precede C17. C21 usa O da C15. C23 usa perm da C21. Da kernel nuovo per permutazione e posizione: C03, C13, C15, C21, C23. Non usare il contatore In[] come riferimento.

**Prima dei calcoli.** Una feature è una componente della rappresentazione. Il backbone la estrae dall’immagine e la testa produce gli score di classe s; qui studiamo gli operatori interni, senza addestrare un classificatore. B conta immagini, C canali, H/W dimensioni spaziali, P lato della patch, N patch, T token inclusa eventuale CLS, D feature per token, h head. Z indica la sequenza delle feature; gli score di attenzione sono S, distinti dagli score di classe. CPU è sufficiente; per Colab seguire Avvio_Colab.md e salvare i risultati prima di chiudere il runtime.

Gli spunti N01–N15 sono accompagnati dalla lettura del risultato. Non occorre compilare campi o preparare una relazione per la Lezione 13. 

Per la sola demo fotografica da kernel nuovo: C29, C31, C33, C35. Il pacchetto include foto e pesi; la precisione è float32, separata dagli esempi assegnati in float64.

In [ ]:
# C03
import math, json, platform
from pathlib import Path
import numpy as np
import torch
from torch import nn
import torch.nn.functional as F
torch.manual_seed(20261026)
torch.set_num_threads(1)
torch.use_deterministic_algorithms(True)
torch.set_default_dtype(torch.float64)
risultati={"seed":20261026,"torch":torch.__version__,"numpy":np.__version__,"python":platform.python_version(),"training":False}
print("CPU, float64, nessun training. PyTorch", torch.__version__)

<!-- C04 -->
## Residuo e gradiente
La shortcut riusa l’ingresso e lo somma alla correzione F(x): y=x+F(x). Il gradiente rispetto a x misura come varia la loss L al variare di x; nel ritorno attraverso una somma si sommano i contributi di tutti i rami che usano x. In questa cella il calcolo termina alla somma y, senza una ReLU successiva. Non si esegue alcun aggiornamento dei pesi.

**N01 · Residuo e confronto con il ramo puro.**

**Lettura guidata.** Per `F(x)=−0.2x` e `x=(1, 2)`, la somma dà `y=(0.8, 1.6)`, loss `1.6` e gradiente `(0.64, 1.28)`. Il gradiente che arriva alla somma è `g_y=y`: il percorso diretto restituisce `(0.8, 1.6)`, quello attraverso F restituisce `(−0.16, −0.32)` e la somma dà il risultato di C05. Nel ramo puro l’uscita è `(−0.2, −0.4)`, la loss `0.1` e il gradiente `(0.04, 0.08)`. Cambia anche la loss: questi numeri non provano che una shortcut aumenti sempre ogni gradiente.

La cella seguente è completa e verifica questo esempio; non richiede modifiche al codice.

In [ ]:
# C05
x=torch.tensor([1.,2.],requires_grad=True)
y=x-0.2*x
loss=0.5*y.square().sum()
g=torch.autograd.grad(loss,x)[0]
x0=x.detach().clone().requires_grad_()
plain=-0.2*x0
plain_loss=0.5*plain.square().sum()
g_plain=torch.autograd.grad(plain_loss,x0)[0]
assert torch.allclose(g,torch.tensor([.64,1.28]))
risultati["residuo"]={"y":y.detach().tolist(),"loss":loss.item(),"grad":g.tolist(),"plain_grad":g_plain.tolist()}
print(json.dumps(risultati["residuo"],indent=2))
print("Jacobiano residuale=0.8 I; puro=-0.2 I. Il gradiente dipende anche dalla loss.")

<!-- C06 -->
## Cambio di forma e proiezione
Una somma elemento per elemento richiede la stessa forma. La proiezione è una trasformazione appresa che adegua canali e griglia; un kernel 1×1 combina canali nella stessa posizione, mentre il passo (stride) sceglie ogni quante posizioni applicarlo. I pesi sono condivisi fra immagini e posizioni.

**N02 · Forma e parametri della proiezione.**

**Lettura guidata.** La proiezione `Conv2d(64, 128, 1, stride=2, bias=False)` porta `(B, 64, 28, 28)` a `(B, 128, 14, 14)`, consentendo la somma con il ramo assegnato. Usa `64*128=8192` pesi. Batch e posizioni non moltiplicano il numero dei parametri, perché gli stessi coefficienti sono condivisi. C07 verifica la forma con `B=2`.

La cella seguente è completa e verifica questo esempio; non richiede modifiche al codice.

In [ ]:
# C07
projection=nn.Conv2d(64,128,kernel_size=1,stride=2,bias=False)
u=torch.zeros(2,64,28,28)
v=projection(u)
count=sum(p.numel() for p in projection.parameters())
assert tuple(v.shape)==(2,128,14,14) and count==8192
risultati["proiezione"]={"shape":list(v.shape),"pesi":count}
print(risultati["proiezione"])

<!-- C08 -->
## Depthwise e pointwise
**Depthwise:** un filtro spaziale distinto per ogni canale, senza mescolare canali. **Pointwise:** una trasformazione 1×1 che combina i canali nella stessa posizione. r è il fattore di espansione dei canali intermedi. Il blocco ConvNeXt comprende anche LayerNorm e GELU; GELU è una funzione deterministica non lineare e liscia. Il conto seguente riguarda solo i pesi delle convoluzioni, non bias o normalizzazioni.

**N03 · Che cosa conta il confronto ConvNeXt.**

**Lettura guidata.** La convoluzione piena usa `7*7*64*64=200704` pesi; la depthwise `7*7*64=3136`; le due pointwise `64*256+256*64=32768`. Il totale dei soli pesi è `35904`, circa `5.590018` volte meno della convoluzione piena. Non è una sostituzione equivalente: mescolamento spaziale e fra canali sono separati e il blocco comprende una non linearità. Il conto di C09 esclude bias, LayerNorm e layer scale.

La cella seguente è completa e verifica questo esempio; non richiede modifiche al codice.

In [ ]:
# C09
C=64; k=7; r=4
full=k*k*C*C
depthwise=k*k*C
pointwise=C*(r*C)+(r*C)*C
risultati["convnext"]={"full":full,"depthwise":depthwise,"pointwise":pointwise,"totale_soli_pesi":depthwise+pointwise,"rapporto":full/(depthwise+pointwise)}
assert (full,depthwise,pointwise)==(200704,3136,32768)
print(risultati["convnext"])
print("Esclusi bias, LayerNorm, layer scale. La composizione include una non linearità e non è equivalente alla conv piena.")

<!-- C10 -->
## Patch embedding
Una **patch** è una regione dell’immagine. Appiattirla significa elencarne ordinatamente i valori; una proiezione affine condivisa produce un **token**, vettore di D feature. Questo passaggio è chiamato patch embedding. Il token CLS è un vettore appreso aggiunto alla sequenza, distinto da una patch e dall’etichetta target: separare quindi N patch e T token totali.

**N04 · Dalla patch al token.**

**Lettura guidata.** A `224×224`, `P=16` produce `196` patch e `197` token con CLS; `P=8` produce `784` patch e `785` token con CLS. Nell’esempio piccolo, la prima patch 2×2 elenca `(0, 1, 4, 5)` e la proiezione assegnata produce il token `(0.1, 1.2, 9.3)`. C11 confronta gli stessi pesi applicati con unfold+Linear e con Conv2d: l’errore massimo nel riferimento è `0`. CLS aggiunge una posizione appresa, non una nuova patch dell’immagine.

La cella seguente è completa e verifica questo esempio; non richiede modifiche al codice.

In [ ]:
# C11
im=torch.arange(16,dtype=torch.float64).reshape(1,1,4,4)
weight=torch.tensor([[1.,0,0,0],[0,1,0,0],[0,0,1,1]])
bias=torch.tensor([.1,.2,.3])
patch=F.unfold(im,kernel_size=2,stride=2).transpose(1,2)
token=F.linear(patch,weight,bias)
conv=F.conv2d(im,weight.reshape(3,1,2,2),bias,stride=2).flatten(2).transpose(1,2)
assert torch.allclose(token,conv)
risultati["patch"]={"patch":patch.tolist(),"token":token.tolist(),"max_error":(token-conv).abs().max().item(),"N16":196,"N8":784,"parametri_D768":(16*16*3+1)*768}
print("Patch in ordine di lettura",patch)
print("Token",token)
print("Conv2d e unfold+Linear: errore massimo",risultati["patch"]["max_error"])

<!-- C12 -->
## Query, key e value assegnati
Una **query** avvia un confronto, una **key** fornisce il vettore con cui confrontarla, un **value** è il contenuto da combinare. Nel modello questi vettori sono proiezioni apprese dei token; qui Q/K/V sono assegnati direttamente per separare i passaggi. Ogni riga rappresenta una posizione. dₖ è il numero di componenti di query/key e dᵥ quello dei value; entrambi valgono 2 nell’esempio. S contiene gli score di compatibilità prima della softmax, distinti dagli score finali di classe s.

**N05 · Compatibilità e contenuto hanno ruoli diversi.**

**Lettura guidata.** Per la prima query, i tre prodotti scalari con le key sono `(1, 0, 1)`. Dividendo per `sqrt(2)` si ottiene la prima riga degli score `S≈(0.707107, 0, 0.707107)`. S descrive compatibilità; V contiene le componenti da combinare. Q, K e V sono assegnati per il calcolo: non sono rappresentazioni apprese su immagini in questo notebook.

La cella seguente è completa e verifica questo esempio; non richiede modifiche al codice.

In [ ]:
# C13
Q=torch.tensor([[1.,0.],[0.,1.],[1.,1.]])
K=torch.tensor([[1.,0.],[0.,1.],[1.,1.]])
V=torch.tensor([[1.,0.],[0.,2.],[3.,1.]])
S=Q@K.T/math.sqrt(Q.shape[-1])
print("Q",Q,"\nK",K,"\nV",V,"\nscore scalati",S)

<!-- C14 -->
## Softmax per riga e combinazione
Per ogni query, la softmax trasforma gli score di una riga in pesi A non negativi con somma 1. L’asse è quello delle key, non quello delle query. L’output O=AV combina i value componente per componente: ogni query produce un nuovo vettore di feature. Le matrici non sono probabilità delle classi Digits.

**N06 · Una riga di attenzione e il suo output.**

**Lettura guidata.** La prima riga di A è circa `(0.401112, 0.197776, 0.401112)`. I contributi dei value sono circa `(0.401112, 0)`, `(0, 0.395552)` e `(1.203336, 0.401112)`; la loro somma dà `o1≈(1.604448, 0.796664)`. Le righe sommano a 1; le colonne sommano circa `(0.847143, 0.847143, 1.305714)`. La normalizzazione usa un denominatore per riga: A può non essere simmetrica anche con Q=K. C15 confronta questo calcolo con scaled_dot_product_attention senza dropout.

La cella seguente è completa e verifica questo esempio; non richiede modifiche al codice.

In [ ]:
# C15
A=torch.softmax(S,dim=-1)
O=A@V
ref=F.scaled_dot_product_attention(Q[None,None],K[None,None],V[None,None],dropout_p=0.0)[0,0]
assert torch.allclose(A.sum(-1),torch.ones(3)) and torch.allclose(O,ref)
risultati["attenzione"]={"Q":Q.tolist(),"K":K.tolist(),"V":V.tolist(),"S":S.tolist(),"A":A.tolist(),"O":O.tolist(),"column_sums":A.sum(0).tolist(),"symmetric":torch.allclose(A,A.T),"sdpa_max_error":(ref-O).abs().max().item()}
print("A",A,"\nO",O,"\nSomme colonne",A.sum(0))
print("A simmetrica?",torch.allclose(A,A.T))

<!-- C16 -->
## Scala degli score prima della softmax

**Entropia e temperatura.** Entropia misura la dispersione dei pesi: $H=-\sum_j A_{ij}\log A_{ij}$ (log naturale, nat). La temperatura $\tau>0$ divide gli score prima della softmax: score moltiplicati per 8 corrispondono a $\tau=1/8$. Un pareggio esatto resta tale.

Il termine logit indica uno score prima della softmax. Una temperatura positiva τ cambia i pesi in softmax(S/τ); τ più piccola amplifica le differenze, mantenendo eventuali pareggi. L’entropia H=−Σ A log A misura quanto sono dispersi i pesi di una riga; qui il logaritmo è naturale e l’unità è il nat.

**N07 · Concentrazione e correttezza sono distinte.**

**Lettura guidata.** Moltiplicare Q per 8 moltiplica S per 8 e corrisponde alla temperatura `tau=1/8`. Le entropie per riga passano circa da `(1.053363, 1.053363, 1.037277)` a `(0.704756, 0.704756, 0.046213)` nat. Le prime due righe conservano due massimi uguali; la terza si concentra quasi su un solo value. C17 misura la dispersione dei pesi, senza un target o una metrica che ne dimostri la correttezza.

La cella seguente è completa e verifica questo esempio; non richiede modifiche al codice.

In [ ]:
# C17
A_hot=torch.softmax(8*S,dim=-1)
entropy=-(A*A.log()).sum(-1)
entropy_hot=-(A_hot*A_hot.log()).sum(-1)
risultati["scala"]={"entropy":entropy.tolist(),"entropy_logits_x8":entropy_hot.tolist(),"A_logits_x8":A_hot.tolist()}
print(risultati["scala"])

<!-- C18 -->
## Multihead
Una **head** è un ramo di attenzione con proiezioni proprie. Più head lavorano in parallelo; concatenare significa affiancare le loro feature per ciascun token. La proiezione finale combina queste componenti, senza aggiungere posizioni alla sequenza. Nel caso standard dₖ=dᵥ=D/h.

**N08 · Una head non aggiunge posizioni.**

**Lettura guidata.** Con `B=2`, `T=5`, `D=12` e `h=3`, ogni head usa 4 componenti. Q/K/V separati per head hanno forma `(2, 3, 5, 4)`, A ha forma `(2, 3, 5, 5)` e l’output concatenato, poi proiettato, ha forma `(2, 5, 12)`. I 624 parametri del modulo includono i bias delle quattro proiezioni. I cinque token restano cinque: si concatenano le feature delle head, non le posizioni.

La cella seguente è completa e verifica questo esempio; non richiede modifiche al codice.

In [ ]:
# C19
mha=nn.MultiheadAttention(embed_dim=12,num_heads=3,batch_first=True,dropout=0.)
z=torch.randn(2,5,12)
out,att=mha(z,z,z,need_weights=True,average_attn_weights=False)
assert tuple(out.shape)==(2,5,12) and tuple(att.shape)==(2,3,5,5)
risultati["multihead"]={"input":list(z.shape),"qkv_head":[2,3,5,4],"attention":list(att.shape),"output":list(out.shape),"params_con_bias":sum(p.numel()for p in mha.parameters())}
print(risultati["multihead"])

<!-- C20 -->
## Permutazione dei token senza posizione
Una permutazione riordina le righe; l’equivarianza significa che riordinare l’ingresso produce lo stesso riordinamento dell’uscita. L’invarianza significa invece che una quantità rimane identica. Il pooling medio calcola la media delle feature lungo le posizioni. Questo esperimento usa attenzione senza maschera, dropout o posizione.

**N09 · Equivarianza senza posizione.**

**Lettura guidata.** Riordinando insieme le righe di Q, K e V, l’output si riordina nello stesso modo: `Op=O[perm]`. La media lungo i token è invece invariata. I due errori sono `0` nel caso verificato in C21. La proprietà richiede le ipotesi dichiarate: niente posizione, maschere dipendenti dallo slot o dropout che alteri il confronto.

La cella seguente è completa e verifica questo esempio; non richiede modifiche al codice.

In [ ]:
# C21
perm=torch.tensor([2,0,1])
Op=torch.softmax(Q[perm]@K[perm].T/math.sqrt(2),-1)@V[perm]
assert torch.allclose(Op,O[perm])
risultati["permutazione"]={"errore_equivarianza":(Op-O[perm]).abs().max().item(),"errore_media":(Op.mean(0)-O.mean(0)).abs().max().item()}
print(risultati["permutazione"])

<!-- C22 -->
## Posizione associata allo slot
Uno slot è una posizione nella sequenza; E assegna un vettore a ciascuno slot. Qui E non è la matrice essenziale della geometria stereo. Permutare soltanto i contenuti e lasciare E fermo significa assegnare contenuti nuovi alle posizioni esistenti. La funzione di codice self_att(z) riceve feature: z non indica score finali di classe.

**N10 · Contenuto e posizione possono essere riordinati diversamente.**

**Lettura guidata.** In C23 i contenuti vengono permutati e i vettori E restano negli slot originali: in generale `perm(Q+E)` e `Q[perm]+E` non coincidono. L’errore massimo rispetto al solo riordinamento dell’uscita è circa `0.3562215413` nel caso assegnato. Se si permutassero insieme contenuti e vettori di posizione, si otterrebbe nuovamente il riordinamento dell’intera sequenza. Il confronto riguarda questa costruzione, senza un risultato di classificazione.

La cella seguente è completa e verifica questo esempio; non richiede modifiche al codice.

In [ ]:
# C23
E=torch.tensor([[0.,0.],[.2,-.1],[-.3,.4]])
def self_att(z):return torch.softmax(z@z.T/math.sqrt(2),-1)@z
base=self_att(Q+E)
shuffled=self_att(Q[perm]+E)
error=(shuffled-base[perm]).abs().max().item()
assert error>1e-6
risultati["posizione"]={"errore_con_posizioni_fisse":error}
print(risultati["posizione"])

<!-- C24 -->
## Costo dell’attenzione
Un MAC (multiply–accumulate) conta un prodotto con accumulo, senza convertirlo in un tempo. A ha h×T×T coefficienti per immagine: ogni head confronta tutte le coppie di token. Un valore float32 occupa 4 byte; 1 MiB equivale a 2²⁰ byte. Separare coefficienti, memoria e operazioni.

**N11 · Coefficienti, memoria e tempo sono quantità diverse.**

**Lettura guidata.** Con CLS e 12 head, `T=197` produce `465708` coefficienti di A; `T=785` ne produce `7394700`. Solo questi coefficienti, se materializzati in float32, occupano circa `1.776535` e `28.208542` MiB. C25 riporta separatamente MAC per QKV/output, attenzione e MLP: nessuno di questi conteggi è una latenza misurata. Sono esclusi gradienti e altre attivazioni; kernel efficienti possono evitare di conservare l’intera matrice A.

La cella seguente è completa e verifica questo esempio; non richiede modifiche al codice.

In [ ]:
# C25
cost=[]
for P in [16,8]:
    T=(224//P)**2+1
    a=12*T*T
    qkv_out=4*T*768**2
    attn=2*T*T*768
    mlp=8*T*768**2
    cost.append({"P":P,"T":T,"coefficienti_A":a,"MiB_A_float32":4*a/2**20,"MAC_QKV_output":qkv_out,"MAC_QK_AV":attn,"MAC_MLP_r4":mlp})
risultati["costo"]=cost
print(json.dumps(cost,indent=2))
print("Rapporto coefficienti",cost[1]["coefficienti_A"]/cost[0]["coefficienti_A"])

<!-- C26 -->
## Sintesi
**N12 · Lettura critica.** Una attention map è una segmentazione o una spiegazione causale della decisione?

**Lettura guidata.** Una attention map mostra i pesi di uno specifico layer e di una specifica head. Non è automaticamente una segmentazione né una spiegazione causale dello score di classe. Un controllo separato può confrontare occlusioni regionali con controlli di pari area, su più immagini, misurando la variazione dello score s a checkpoint e modalità fissi. Anche questa prova ha limiti: l’occlusione può creare input fuori distribuzione. Qui si discute il metodo; non si assegnano nuovi esperimenti.

### Sintesi svolta F01–F05

- **F01 — Residuo N01:** y=(0.8, 1.6), L=1.6 e gradiente (0.64, 1.28), con somma del percorso diretto e del ramo F.
- **F02 — Attenzione N06:** la prima riga di A è circa (0.401112, 0.197776, 0.401112); la somma pesata dei value produce circa (1.604448, 0.796664).
- **F03 — Costo N11:** con CLS, T=197/785 e 12 head, A contiene 465708/7394700 coefficienti, circa 1.776535/28.208542 MiB in float32, con le esclusioni dichiarate in C24.
- **F04 — Interpretazione:** patch più piccole aumentano fortemente il termine quadratico dell’attenzione; questo conto non determina da solo la latenza di un modello su un dispositivo.
- **F05 — Consultazione:** C27 salva il JSON dei piccoli esempi; C35 lo aggiorna con la demo fotografica C28–C35, se eseguita. Se utile al ripasso, si può conservare una copia personale del notebook e dei risultati; conservarli per lo studio.

Questa sintesi è già disponibile: non si richiede una pagina aggiuntiva. Per leggere la successiva Lezione 14, distinguere preprocessing, architettura, checkpoint e modalità del modello preaddestrato; le istruzioni dell’elaborato individuale sono nel notebook della Lezione 14.

In [ ]:
# C27
assert not risultati["training"]
Path("risultati_lezione13.json").write_text(json.dumps(risultati,indent=2),encoding="utf-8")
print("Salvati risultati_lezione13.json; nessuna metrica su dataset e nessun modello addestrato.")

<!-- C28 -->
## Una CLS-verso-patch su una fotografia

Questa demo riprende **S55–S56 e §13.18 della dispensa**. Usiamo DeiT Tiny **non distillato**, con pesi ImageNet originali: 12 blocchi, 3 head, embedding di 192 componenti. A 224×224 pixel, patch 16×16 producono 196 token di patch e un token CLS, in tutto 197 token.

**N13 · Quale parte della matrice di attenzione rappresenta CLS-verso-patch?** Prima individuare l'asse delle query e quello delle key; poi leggere la selezione in C31. Il modulo di supporto esegue il forward del modello e restituisce la matrice dell'ultimo blocco. L'estrazione e i calcoli della mappa restano visibili qui.

**Preparazione prima dell'esecuzione.** C29 verifica i file locali e carica il modello su CPU. Eseguire C31 e C33. Da kernel nuovo bastano **C29, C31, C33, C35** per questa sola demo. I tre passaggi precedenti e gli ID C01–C27 restano validi.

La foto *Chelsea* è di Stefan van der Walt, CC0, da [scikit-image](https://scikit-image.org/docs/stable/api/skimage.data.html#skimage.data.chelsea). [DeiT, Touvron et al., ICML 2021](https://proceedings.mlr.press/v139/touvron21a.html) documenta il modello. Questo esempio è un'inferenza, non un risultato di training del notebook.

In [ ]:
# C29
import sys, json, platform
from pathlib import Path
import numpy as np
import torch
import matplotlib.pyplot as plt
from matplotlib import colors, ticker
from matplotlib.patches import Rectangle

# I file devono essere nella stessa cartella di lavoro o nello ZIP estratto.
cls_candidati = [
    Path.cwd(),
    Path.cwd() / "Dalle_CNN_ai_transformer_studenti",
    Path.cwd().parent,
    Path.cwd().parent.parent / "notebook",
    Path.cwd() / "Lezioni/13_2026-10-26_Dalle_CNN_ai_transformer/notebook",
]
cls_base = next((p for p in cls_candidati
                 if (p / "supporto/cls_attention.py").is_file()), None)
if cls_base is None:
    raise FileNotFoundError(
        "Manca supporto/cls_attention.py. Estrai lo ZIP completo e segui Avvio_Colab.md; "
        "il solo file .ipynb non contiene foto e pesi."
    )
sys.path.insert(0, str(cls_base / "supporto"))
from cls_attention import prepara_demo, calcola_attention

torch.set_num_threads(1)
cls_model, cls_input, cls_rgb, cls_info = prepara_demo(
    cls_base / "input/chelsea.png",
    cls_base / "pesi/deit_tiny_patch16_224-a1311bcf.pth",
)
cls_out = Path("output_cls")
cls_out.mkdir(exist_ok=True)
if "risultati" not in globals():
    risultati = {"torch": torch.__version__, "numpy": np.__version__,
                 "python": platform.python_version(), "training": False}
print("Modello pronto: CPU, float32, eval; nessun addestramento.")
print("Input:", tuple(cls_input.shape), "dtype:", cls_input.dtype)
print("Dtype generale del notebook:", torch.get_default_dtype())
print("Risorse locali: supporto/cls_attention.py, input/chelsea.png e checkpoint DeiT.")


<!-- C30 -->
## La riga della query CLS

Il tensore `cls_attn` ha assi **(batch, head, query, key)** e forma **(1, 3, 197, 197)**. L'indice 0 della sequenza è CLS. Le patch occupano gli indici da 1 a 196, per righe dell'immagine.

**Risposta N13.** `cls_attn[0, :, 0, 1:]` sceglie la prima immagine, tutte le head, la query CLS e le sole colonne delle patch. Si ottengono 3 righe di 196 pesi. La colonna CLS completa risponderebbe a un'altra domanda: come tutte le query pesano il value di CLS.

Per ogni head, la riga completa somma a 1. Escludere CLS-verso-CLS lascia una massa pari a $1-A_{0,0}$, in generale minore di 1. Per questa figura **mediamo prima i pesi grezzi delle tre head e poi rinormalizziamo sulle patch**:

$$
\bar a_j=\frac{1}{3}\sum_{\alpha=1}^{3}A^{(12,\alpha)}_{0,j},\qquad
\widetilde a_j=\frac{\bar a_j}{\sum_{k=1}^{196}\bar a_k}.
$$

Il forward usa `eval` e `inference_mode`: dropout disattivato e nessun grafo per il backward. Il modello e l'input restano float32 anche quando C03 ha impostato float64 per i piccoli esempi analitici. Non si modificano pesi o dati del test.

In [ ]:
# C31
# Forward del modello preaddestrato; A è dopo la softmax, prima di A @ V.
cls_attn = calcola_attention(cls_model, cls_input)

# Righe = query, colonne = key. Il token speciale non ha un pixel nell'immagine.
cls_per_head = cls_attn[0, :, 0, 1:]       # (3, 196)
cls_a00 = cls_attn[0, :, 0, 0]           # CLS-verso-CLS, per head
cls_mean = cls_per_head.mean(dim=0)      # prima la media dei pesi grezzi
cls_patch_mass = cls_mean.sum()
cls_weights = cls_mean / cls_patch_mass # poi la normalizzazione sulle patch
cls_grid = cls_weights.reshape(14, 14)  # ordine per righe, come le patch

assert tuple(cls_attn.shape) == (1, 3, 197, 197)
assert torch.allclose(cls_patch_mass, 1 - cls_a00.mean(), atol=3e-7, rtol=0)
assert torch.allclose(cls_weights.sum(), torch.ones_like(cls_weights.sum()),
                      atol=3e-7, rtol=0)
cls_index = int(cls_weights.argmax())
cls_row0, cls_col0 = divmod(cls_index, 14)
print("Tensore A:", tuple(cls_attn.shape), "CLS-verso-patch:", tuple(cls_per_head.shape))
print(f"CLS-verso-CLS medio: {cls_a00.mean().item():.6f}")
print(f"Massa grezza sulle patch: {cls_patch_mass.item():.6f}")
print(f"Somma dopo normalizzazione: {cls_weights.sum().item():.6f}")
print(f"Massimo: riga {cls_row0 + 1}, colonna {cls_col0 + 1}, "
      f"patch {cls_index + 1}, indice Python {cls_index}, peso {cls_weights.max().item():.6f}")


<!-- C32 -->
## Immagine, attention map e sovrapposizione

**N14 · Come si passa dai 196 pesi alla mappa e dove cade il massimo?** Il `reshape(14, 14)` conserva l'ordine per righe delle patch. Numerando righe e colonne da 1, il riferimento è **riga 7, colonna 10**, cioè patch $(7-1)\cdot14+10=94$, indice Python 93. Il peso normalizzato è circa **0.095411**; il suo coefficiente grezzo medio era circa **0.071510**. La massa grezza complessiva sulle patch è circa **0.749493**, perché CLS conserva il peso restante.

C33 mostra il ritaglio realmente fornito al modello, la griglia 14×14 e la sovrapposizione. Una cornice turchese segnala il massimo. La stessa scala cromatica vale per mappa e sovrapposizione: chiaro significa peso maggiore. I valori si quantificano sulla barra della **mappa**, perché sulla fotografia i colori si mescolano con quelli dell'immagine.

Ogni peso viene replicato su una patch 16×16, con interpolazione al vicino più prossimo e opacità 0.48. L'ingrandimento non aggiunge osservazioni spaziali. C33 salva anche `output_cls/cls_attention_chelsea.png`.

In [ ]:
# C33
cls_grid_np = cls_grid.cpu().numpy()
cls_fig, cls_axes = plt.subplots(1, 3, figsize=(13.5, 4.6), dpi=150)
cls_norm = colors.Normalize(vmin=0, vmax=float(cls_grid_np.max()))
cls_cmap = "magma"

cls_axes[0].imshow(cls_rgb)
for edge in range(0, 225, 16):
    cls_axes[0].axhline(edge - 0.5, color="white", lw=0.45, alpha=0.65)
    cls_axes[0].axvline(edge - 0.5, color="white", lw=0.45, alpha=0.65)
cls_axes[0].set_title("Immagine e patch", fontsize=17, fontweight="bold")

cls_im = cls_axes[1].imshow(
    cls_grid_np, cmap=cls_cmap, norm=cls_norm, interpolation="nearest",
    extent=(0.5, 14.5, 14.5, 0.5),
)
cls_axes[1].set_title("Pesi CLS → patch", fontsize=17, fontweight="bold")
cls_axes[1].set_xticks([1, 7, 10, 14])
cls_axes[1].set_yticks([1, 7, 14])
cls_axes[1].set_xlabel("Colonna", fontsize=14)
cls_axes[1].set_ylabel("Riga", fontsize=14)
cls_axes[1].add_patch(Rectangle(
    (cls_col0 + 0.5, cls_row0 + 0.5), 1, 1,
    fill=False, edgecolor="#00e7da", lw=2.4,
))

cls_nearest = np.repeat(np.repeat(cls_grid_np, 16, axis=0), 16, axis=1)
cls_axes[2].imshow(cls_rgb)
cls_axes[2].imshow(cls_nearest, cmap=cls_cmap, norm=cls_norm,
                   alpha=0.48, interpolation="nearest")
cls_axes[2].add_patch(Rectangle(
    (cls_col0 * 16 - 0.5, cls_row0 * 16 - 0.5), 16, 16,
    fill=False, edgecolor="#00e7da", lw=2.4,
))
cls_axes[2].set_title("Sovrapposizione", fontsize=17, fontweight="bold")
for ax in (cls_axes[0], cls_axes[2]):
    ax.set_xticks([])
    ax.set_yticks([])
for ax in cls_axes:
    ax.tick_params(labelsize=12)
    for spine in ax.spines.values():
        spine.set_visible(False)
cls_fig.subplots_adjust(left=0.015, right=0.875, bottom=0.19, top=0.86, wspace=0.28)
cls_cax = cls_fig.add_axes([0.903, 0.25, 0.014, 0.55])
cls_bar = cls_fig.colorbar(cls_im, cax=cls_cax)
cls_bar.ax.yaxis.set_major_formatter(ticker.FormatStrFormatter("%.3f"))
cls_bar.ax.tick_params(labelsize=11)
cls_bar.set_label("Peso sulle patch", fontsize=13, labelpad=10)
cls_fig.text(0.5, 0.045,
             "DeiT Tiny · blocco 12 · media delle 3 head · foto Chelsea: Stefan van der Walt, CC0",
             ha="center", fontsize=11)
cls_png = cls_out / "cls_attention_chelsea.png"
cls_fig.savefig(cls_png, dpi=180, facecolor="white")
plt.show()
plt.close(cls_fig)
print("Figura salvata:", cls_png)


<!-- C34 -->
## Lettura critica della mappa

**N15 · Il peso 0.095411 è una probabilità di classe? Che cosa dimostra la figura?**

**Risposta.** Il valore è circa il **9.54% della massa mostrata sulle patch**, dopo aver escluso CLS e rinormalizzato. Non è la probabilità che quella regione contenga un gatto, né il suo contributo causale allo score della classe. Il massimo è vicino all'angolo interno dell'occhio a destra per chi guarda. Un altro peso elevato si trova a riga 12, colonna 3, nella zona dei baffi e della guancia a sinistra.

La distribuzione è disomogenea e non produce una sagoma continua. Tutti i pesi di questo esempio sono positivi: il colore scuro indica un peso piccolo. Inoltre, nell'ultimo blocco ogni value contiene già contesto mescolato dai blocchi precedenti. La media delle tre head non è un attention rollout, che combina transizioni su più blocchi con ulteriori ipotesi. Per studiare una relazione causale servirebbero interventi e controlli; un'occlusione può introdurre immagini fuori distribuzione.

Questa è una dimostrazione già svolta, senza nuovi esperimenti da consegnare. C35 conserva i pesi e aggiorna `risultati_lezione13.json`, che C27 aveva salvato prima della demo fotografica. Da una ripartenza parziale il JSON contiene soltanto i blocchi effettivamente eseguiti.

Fonti: [visualizzazione ufficiale DINO](https://github.com/facebookresearch/dino/blob/main/visualize_attention.py), per l'estrazione CLS, e [Abnar e Zuidema, ACL 2020](https://aclanthology.org/2020.acl-main.385/), per la distinzione dal rollout. La figura qui usa DeiT supervisionato e non attribuisce al modello i risultati di DINO.

In [ ]:
# C35
cls_npz = cls_out / "cls_attention_misure.npz"
np.savez_compressed(
    cls_npz,
    attention_last_block=cls_attn.cpu().numpy(),
    cls_patch_per_head=cls_per_head.cpu().numpy(),
    cls_patch_head_mean=cls_mean.cpu().numpy(),
    cls_patch_normalized=cls_weights.cpu().numpy(),
    cls_patch_grid=cls_grid_np,
    input_normalized=cls_input.cpu().numpy(),
    input_rgb=cls_rgb,
)
risultati["cls_patch"] = {
    **cls_info,
    "A00_per_head": cls_a00.cpu().tolist(),
    "A00_head_mean": cls_a00.mean().item(),
    "patch_mass_head_mean": cls_patch_mass.item(),
    "normalized_patch_sum": cls_weights.sum().item(),
    "normalized_patch_min": cls_weights.min().item(),
    "normalized_patch_max": cls_weights.max().item(),
    "argmax_row_col_1_based": [cls_row0 + 1, cls_col0 + 1],
    "argmax_patch_index_1_based": cls_index + 1,
    "operation": "media pesi grezzi di 3 head, poi rinormalizzazione sulle patch",
    "upsampling": "nearest", "overlay_alpha": 0.48, "rollout": False,
    "figure": str(cls_png), "arrays": str(cls_npz),
}
assert not risultati["training"] and not risultati["cls_patch"]["training"]
Path("risultati_lezione13.json").write_text(
    json.dumps(risultati, ensure_ascii=False, indent=2), encoding="utf-8"
)
print("Salvati:", cls_npz, "e", "risultati_lezione13.json")
print("Blocchi presenti:", ", ".join(k for k, v in risultati.items() if isinstance(v, (dict, list))))
